# DroneAI Stage 0 gate

This notebook verifies the Colab GPU, private Git checkout, Google Drive persistence, tests, and reproducibility metadata. It does not train a model.

Before running, add a Colab secret named `GITHUB_TOKEN`, grant this notebook access, and select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/DroneAI')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)

In [ ]:
import os, stat, subprocess
from google.colab import userdata

REPO_URL = 'https://github.com/LuciTa81/DroneAI.git'
REPO_DIR = Path('/content/DroneAI')
token = userdata.get('GITHUB_TOKEN')
if not token:
    raise RuntimeError('Add the GITHUB_TOKEN secret and grant notebook access.')

askpass = Path('/tmp/droneai_git_askpass.py')
askpass.write_text(
    "#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1].lower() if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'password' in prompt else 'x-access-token')\n",
    encoding='utf-8',
)
askpass.chmod(askpass.stat().st_mode | stat.S_IEXEC)
git_env = os.environ.copy()
git_env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': token})
try:
    if (REPO_DIR / '.git').is_dir():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], env=git_env, check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], env=git_env, check=True)
finally:
    askpass.unlink(missing_ok=True)
    del token
    git_env.pop('GITHUB_TOKEN', None)

subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], cwd=REPO_DIR, check=True)

In [ ]:
result = subprocess.run(
    [sys.executable, 'scripts/run_stage0.py', '--drive-root', str(DRIVE_ROOT), '--repo-root', str(REPO_DIR)],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
print('Exit code:', result.returncode)

In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

report_dir = DRIVE_ROOT / 'runs' / 'stage-0'
score = json.loads((report_dir / 'score.json').read_text(encoding='utf-8'))
display(Markdown((report_dir / 'score.md').read_text(encoding='utf-8')))
display(pd.DataFrame(score['checks'])[['category', 'description', 'weight', 'earned', 'blocker', 'observed']])
print('Decision:', score['status'], score['score'], '/ 100')